# Chapter 2: Binary Channels and Model Order

Companion notebook for **Chapter 2** of *Bernoulli Types: An Algebra of Approximate Computation*.
Source: `book/chapters/ch02/`.

This notebook builds 2x2 channel matrices in numpy, verifies the BSC composition formula from
Section 2.3, plots BSC capacity, explores the (epsilon, omega) parameter space, and visualizes
partition blocks for input domains of size 2, 3, and 4.

**Conventions (matching book notation):**
- `epsilon` (FPR) = false-positive rate
- `omega` (FNR) = false-negative rate
- Channel matrix `Q` is row-stochastic: rows indexed by input block, columns by output label.
  - Row 0 (B_1): positive inputs (true membership)
  - Row 1 (B_2): negative inputs (true non-membership)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize

# Symbolic import for library continuity -- package is in placeholder state;
# we use numpy directly for all channel matrix computations.
try:
    from bernoulli import Bernoulli
    _BERNOULLI_AVAILABLE = True
except ImportError:
    _BERNOULLI_AVAILABLE = False

print(f"NumPy {np.__version__}")
print(f"bernoulli package available: {_BERNOULLI_AVAILABLE}")

## 1. Build Channel Matrices in NumPy

A **binary channel** maps each input (positive or negative) to one of two output labels
(report-positive or report-negative). The channel is fully characterized by its
$2 \times 2$ transition matrix $Q$:

$$Q = \begin{pmatrix} 1 - \omega & \omega \\ \epsilon & 1 - \epsilon \end{pmatrix}$$

where row 0 (B_1, positives) has false-negative rate $\omega$, and row 1 (B_2, negatives)
has false-positive rate $\epsilon$.

The **Binary Symmetric Channel (BSC)** is the special case $\omega = \epsilon$:

$$Q_{\text{BSC}} = \begin{pmatrix} 1 - \epsilon & \epsilon \\ \epsilon & 1 - \epsilon \end{pmatrix}$$

In [ ]:
def make_bsc(epsilon: float) -> np.ndarray:
    """Build a Binary Symmetric Channel matrix.

    Parameters
    ----------
    epsilon : float
        Crossover (error) probability in [0, 1].

    Returns
    -------
    Q : ndarray of shape (2, 2)
        Row-stochastic channel matrix.
        Row 0 (B_1, positives): [1-epsilon, epsilon]
        Row 1 (B_2, negatives): [epsilon,   1-epsilon]
    """
    e = float(epsilon)
    return np.array([[1.0 - e, e],
                     [e,       1.0 - e]])


def make_asymmetric(epsilon: float, omega: float) -> np.ndarray:
    """Build a general asymmetric binary channel matrix.

    Parameters
    ----------
    epsilon : float
        False-positive rate (FPR) in [0, 1].
    omega : float
        False-negative rate (FNR) in [0, 1].

    Returns
    -------
    Q : ndarray of shape (2, 2)
        Row-stochastic channel matrix.
        Row 0 (B_1, positives): [1-omega, omega]
        Row 1 (B_2, negatives): [epsilon, 1-epsilon]
    """
    e, w = float(epsilon), float(omega)
    return np.array([[1.0 - w, w],
                     [e,       1.0 - e]])


def print_matrix(Q: np.ndarray, name: str = "Q") -> None:
    """Pretty-print a 2x2 channel matrix."""
    labels = ["report+", "report-"]
    block_labels = ["B1 (pos)", "B2 (neg)"]
    print(f"{name}:")
    print(f"          {labels[0]:>10s}  {labels[1]:>10s}")
    for i, row_label in enumerate(block_labels):
        print(f"  {row_label}  {Q[i, 0]:10.6f}  {Q[i, 1]:10.6f}")
    print(f"  row sums: {Q.sum(axis=1)}")
    print()


# Verify row-stochasticity
Q_bsc_05 = make_bsc(0.05)
print_matrix(Q_bsc_05, "BSC(epsilon=0.05)")

Q_asym = make_asymmetric(epsilon=0.05, omega=0.10)
print_matrix(Q_asym, "Asymmetric(epsilon=0.05, omega=0.10)")

# Z-channel: omega = 0 (no false negatives)
Q_z = make_asymmetric(epsilon=0.05, omega=0.0)
print_matrix(Q_z, "Z-channel(epsilon=0.05, omega=0.0)")

## 2. Compose Two BSCs

**Serial channel composition** (Section 2.3): input passes through channel $Q_1$, then the output
of $Q_1$ passes through channel $Q_2$. The joint channel is the matrix product:

$$Q_{\text{total}} = Q_2 \cdot Q_1$$

**Note:** This is *serial* composition (matrix product), not *parallel* composition
(Kronecker product used in Chapter 4 for independent queries).

For two BSCs with crossover rates $\epsilon_1$ and $\epsilon_2$, the composition formula
from Section 2.3 gives:

$$\epsilon_{\text{total}} = \epsilon_1(1 - \epsilon_2) + (1 - \epsilon_1)\epsilon_2
= \epsilon_1 + \epsilon_2 - 2\epsilon_1\epsilon_2$$

We verify this algebraically and numerically below.

In [ ]:
def bsc_composition_formula(e1: float, e2: float) -> float:
    """Closed-form crossover for composition of two BSCs.

    From Section 2.3: epsilon_total = e1 + e2 - 2*e1*e2.
    """
    return e1 + e2 - 2.0 * e1 * e2


def compose_channels(Q1: np.ndarray, Q2: np.ndarray) -> np.ndarray:
    """Serial composition of channels Q1 then Q2.

    Q_total = Q2 @ Q1  (matrix product, row-stochastic convention)
    """
    return Q2 @ Q1


# Verify the formula for a specific example
e1, e2 = 0.1, 0.2
Q1 = make_bsc(e1)
Q2 = make_bsc(e2)
Q_comp = compose_channels(Q1, Q2)

e_formula = bsc_composition_formula(e1, e2)
e_numeric = Q_comp[1, 0]  # bottom-left entry is the FPR of the composed BSC

print(f"e1 = {e1}, e2 = {e2}")
print_matrix(Q1, "Q1 (BSC, e1=0.1)")
print_matrix(Q2, "Q2 (BSC, e2=0.2)")
print_matrix(Q_comp, "Q_total = Q2 @ Q1")
print(f"Formula: e_total = e1 + e2 - 2*e1*e2 = {e_formula:.8f}")
print(f"Numeric (Q_comp[1,0]):                 {e_numeric:.8f}")
print(f"Match: {np.isclose(e_formula, e_numeric)}")

In [ ]:
# Plot epsilon_total as a function of epsilon_1, for fixed epsilon_2 = 0.1
e2_fixed = 0.1
e1_vals = np.linspace(0.0, 1.0, 300)
e_total_formula = bsc_composition_formula(e1_vals, e2_fixed)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(e1_vals, e_total_formula, color="steelblue", linewidth=2,
        label=r"$\epsilon_{\mathrm{total}} = \epsilon_1 + \epsilon_2 - 2\epsilon_1\epsilon_2$")
ax.axhline(e2_fixed, color="gray", linestyle="--", linewidth=1,
           label=rf"$\epsilon_2 = {e2_fixed}$ (fixed)")
ax.axvline(e2_fixed, color="tomato", linestyle=":", linewidth=1,
           label=rf"$\epsilon_1 = \epsilon_2 = {e2_fixed}$")
ax.set_xlabel(r"$\epsilon_1$", fontsize=13)
ax.set_ylabel(r"$\epsilon_{\mathrm{total}}$", fontsize=13)
ax.set_title(rf"BSC Composition: $\epsilon_2 = {e2_fixed}$ fixed", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: when epsilon_1 = 0.5, the composed channel is also at 0.5 (pure noise).")
print("When epsilon_1 = 0, the total rate equals epsilon_2 (Q1 is perfect).")
print("The formula is symmetric in e1 and e2.")

## 3. Plot BSC Capacity

The **capacity** of a BSC with crossover probability $\epsilon$ measures the maximum
reliable information transmission rate (in bits per channel use):

$$C(\epsilon) = 1 - H_2(\epsilon)$$

where $H_2(\epsilon) = -\epsilon \log_2 \epsilon - (1-\epsilon) \log_2(1-\epsilon)$
is the binary entropy function.

Key values:
- $C(0) = 1$: perfect channel, full capacity.
- $C(0.5) = 0$: completely noisy channel, zero capacity.
- $C(1) = 1$: completely inverted channel, again full capacity (flip every bit).

In [ ]:
def binary_entropy(p: np.ndarray) -> np.ndarray:
    """Binary entropy H_2(p) = -p log2(p) - (1-p) log2(1-p)."""
    p = np.asarray(p, dtype=float)
    # Avoid log(0) at the boundaries
    result = np.zeros_like(p)
    mask = (p > 0) & (p < 1)
    q = p[mask]
    result[mask] = -q * np.log2(q) - (1.0 - q) * np.log2(1.0 - q)
    return result


def bsc_capacity(epsilon: np.ndarray) -> np.ndarray:
    """BSC capacity C(epsilon) = 1 - H_2(epsilon)."""
    return 1.0 - binary_entropy(epsilon)


eps = np.linspace(0.0, 1.0, 500)
cap = bsc_capacity(eps)
H2 = binary_entropy(eps)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: binary entropy
axes[0].plot(eps, H2, color="darkorange", linewidth=2)
axes[0].set_xlabel(r"$\epsilon$", fontsize=13)
axes[0].set_ylabel(r"$H_2(\epsilon)$ (bits)", fontsize=13)
axes[0].set_title("Binary Entropy Function", fontsize=13)
axes[0].axvline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.6)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# Right: channel capacity
axes[1].plot(eps, cap, color="steelblue", linewidth=2)
axes[1].fill_between(eps, cap, alpha=0.15, color="steelblue")
axes[1].set_xlabel(r"$\epsilon$", fontsize=13)
axes[1].set_ylabel(r"$C(\epsilon)$ (bits/use)", fontsize=13)
axes[1].set_title(r"BSC Capacity: $C(\epsilon) = 1 - H_2(\epsilon)$", fontsize=13)
axes[1].axvline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="epsilon = 0.5 (no info)")
axes[1].annotate("C = 0 at eps = 0.5", xy=(0.5, 0.0), xytext=(0.6, 0.15),
                 arrowprops=dict(arrowstyle="->", color="gray"), fontsize=10, color="gray")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"C(0.00) = {bsc_capacity(np.array([0.001]))[0]:.4f}  (near 1)")
print(f"C(0.05) = {bsc_capacity(np.array([0.05]))[0]:.4f}")
print(f"C(0.10) = {bsc_capacity(np.array([0.10]))[0]:.4f}")
print(f"C(0.50) = {bsc_capacity(np.array([0.50]))[0]:.4f}  (zero capacity)")

## 4. Explore the (epsilon, omega) Parameter Space

For the general asymmetric channel, we have two independent parameters:
- $\epsilon$ (FPR): probability that a negative input is reported positive.
- $\omega$ (FNR): probability that a positive input is reported negative.

We plot two derived quantities over the full parameter space:
1. **Overall accuracy** under uniform input: $\text{acc} = 0.5(1-\omega) + 0.5(1-\epsilon)$
2. **Asymmetry ratio**: $|\epsilon - \omega|$, which is zero on the BSC diagonal.

In [ ]:
# Build a grid over (epsilon, omega)
n_pts = 200
eps_vals = np.linspace(0.0, 1.0, n_pts)
omega_vals = np.linspace(0.0, 1.0, n_pts)
EPS, OMEGA = np.meshgrid(eps_vals, omega_vals)

# Overall accuracy under uniform (50/50) input
acc = 0.5 * (1.0 - OMEGA) + 0.5 * (1.0 - EPS)

# Asymmetry ratio
asym = np.abs(EPS - OMEGA)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy heatmap
im0 = axes[0].contourf(EPS, OMEGA, acc, levels=20, cmap="RdYlGn")
plt.colorbar(im0, ax=axes[0], label="Accuracy")
# Mark the BSC diagonal
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.7, label="BSC (epsilon = omega)")
axes[0].set_xlabel(r"$\epsilon$ (FPR)", fontsize=12)
axes[0].set_ylabel(r"$\omega$ (FNR)", fontsize=12)
axes[0].set_title("Overall Accuracy (uniform input)", fontsize=12)
axes[0].legend(fontsize=9)

# Asymmetry heatmap
im1 = axes[1].contourf(EPS, OMEGA, asym, levels=20, cmap="Blues")
plt.colorbar(im1, ax=axes[1], label=r"$|\epsilon - \omega|$")
axes[1].plot([0, 1], [0, 1], "r--", linewidth=1.5, alpha=0.7, label="BSC diagonal (asymmetry = 0)")
# Mark the Z-channel lines
axes[1].axhline(0.0, color="orange", linestyle=":", linewidth=1.5, alpha=0.8, label="Z-channel (omega = 0)")
axes[1].axvline(0.0, color="green", linestyle=":", linewidth=1.5, alpha=0.8, label="Reverse Z (epsilon = 0)")
axes[1].set_xlabel(r"$\epsilon$ (FPR)", fontsize=12)
axes[1].set_ylabel(r"$\omega$ (FNR)", fontsize=12)
axes[1].set_title(r"Asymmetry: $|\epsilon - \omega|$", fontsize=12)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("The BSC diagonal (epsilon = omega) is where asymmetry = 0.")
print("Bloom filter Z-channel sits at omega = 0 (no false negatives, some false positives).")

## 5. Visualize Partition Blocks

**Model order** (Section 2.4) is the number of partition blocks of the input domain.
Each block gets its own per-block error rate.

For a binary input domain $\{\top, \bot\}$ (size 2), model order is at most 2:
- 1 block: symmetric (BSC) -- same error rate for all inputs.
- 2 blocks: asymmetric -- B_1 (positives) and B_2 (negatives) get independent rates.

We visualize partition block structure for domains of size 2, 3, and 4.
The size-4 domain **previews Chapter 9**, which generalizes to n-ary channels.

In [ ]:
def draw_partition_blocks(ax, domain_size: int, n_blocks: int, colors, title: str,
                          block_labels=None, note: str = "") -> None:
    """Draw a visual of the input domain partitioned into n_blocks blocks."""
    block_size = domain_size // n_blocks
    remainder = domain_size % n_blocks

    x = 0
    for i in range(n_blocks):
        w = block_size + (1 if i < remainder else 0)
        rect = mpatches.FancyBboxPatch(
            (x, 0.2), w, 0.6,
            boxstyle="round,pad=0.05",
            facecolor=colors[i % len(colors)],
            edgecolor="black",
            linewidth=1.5,
            alpha=0.85
        )
        ax.add_patch(rect)
        label = block_labels[i] if block_labels else f"B_{i+1}"
        ax.text(x + w / 2.0, 0.5, label,
                ha="center", va="center", fontsize=10, fontweight="bold")
        x += w

    ax.set_xlim(-0.1, domain_size + 0.1)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=11, pad=6)
    if note:
        ax.text(domain_size / 2.0, 0.06, note, ha="center", va="center",
                fontsize=9, color="dimgray", style="italic")
    ax.axis("off")


COLORS = ["#6baed6", "#fd8d3c", "#74c476", "#9e9ac8"]

fig, axes = plt.subplots(3, 1, figsize=(9, 6))
fig.suptitle("Partition Blocks of the Input Domain", fontsize=13, fontweight="bold", y=0.98)

# Size-2 domain: BSC (1 block, symmetric)
draw_partition_blocks(
    axes[0], domain_size=2, n_blocks=1,
    colors=[COLORS[0]],
    title="Domain size 2, model order 1 (BSC: one shared error rate)",
    block_labels=["B1 = {+, -}"],
    note="epsilon is the same for positives and negatives"
)

# Size-2 domain: Asymmetric (2 blocks, general)
draw_partition_blocks(
    axes[1], domain_size=2, n_blocks=2,
    colors=COLORS[:2],
    title="Domain size 2, model order 2 (Asymmetric: per-block error rates)",
    block_labels=["B1 = {+}", "B2 = {-}"],
    note="omega for B1 (FNR), epsilon for B2 (FPR) -- independent"
)

# Size-4 domain: Chapter 9 preview (4 blocks, one per element)
draw_partition_blocks(
    axes[2], domain_size=4, n_blocks=4,
    colors=COLORS[:4],
    title="Domain size 4 -- PREVIEW: Chapter 9 generalizes to n-ary channels",
    block_labels=["B1", "B2", "B3", "B4"],
    note="Each block has its own per-block error rate; model order up to domain cardinality"
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("Boolean ceiling (Section 2.4): for domain {T, F} (size 2),")
print("model order is at most 2. This is the maximum number of partition blocks.")
print()
print("Preview (Chapter 9): for a domain of size n, model order can be up to n,")
print("generalizing the binary framework to n-ary channels.")

## 6. Partition Blocks for Domain Size 3 (Additional Detail)

For completeness, we show the range of possible partition structures for a
domain of size 3, illustrating that model order is a design choice that can
be coarser than the maximum.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.5))
fig.suptitle("Domain size 3: partition structures by model order", fontsize=12, y=1.02)

draw_partition_blocks(
    axes[0], domain_size=3, n_blocks=1,
    colors=[COLORS[0]],
    title="Order 1: one block",
    block_labels=["B1 = {a,b,c}"],
    note="all elements share one error rate"
)

draw_partition_blocks(
    axes[1], domain_size=3, n_blocks=2,
    colors=COLORS[:2],
    title="Order 2: two blocks",
    block_labels=["B1 = {a,b}", "B2 = {c}"],
    note="two independent error rates"
)

draw_partition_blocks(
    axes[2], domain_size=3, n_blocks=3,
    colors=COLORS[:3],
    title="Order 3: three blocks",
    block_labels=["B1={a}", "B2={b}", "B3={c}"],
    note="each element has its own error rate"
)

plt.tight_layout()
plt.show()

print("Model order is a choice about how finely to partition the input domain.")
print("Finer partitions (higher order) allow more expressive error models")
print("at the cost of more parameters to estimate.")

## 7. Summary and Verification

Quick sanity checks to confirm all key results from Chapter 2 compute correctly.

In [ ]:
print("=" * 60)
print("Chapter 2 Verification Summary")
print("=" * 60)

# 1. BSC matrix is row-stochastic
Q = make_bsc(0.07)
assert np.allclose(Q.sum(axis=1), 1.0), "BSC is not row-stochastic"
print("PASS: BSC(0.07) is row-stochastic")

# 2. Asymmetric matrix is row-stochastic
Q = make_asymmetric(0.05, 0.10)
assert np.allclose(Q.sum(axis=1), 1.0), "Asymmetric channel is not row-stochastic"
print("PASS: Asymmetric(eps=0.05, omega=0.10) is row-stochastic")

# 3. BSC composition formula verified for multiple pairs
test_pairs = [(0.1, 0.2), (0.3, 0.4), (0.05, 0.05), (0.0, 0.3)]
for e1, e2 in test_pairs:
    Q_comp = compose_channels(make_bsc(e1), make_bsc(e2))
    e_formula = bsc_composition_formula(e1, e2)
    # Top-right entry is FNR of composed BSC, also equals FPR by symmetry
    assert np.isclose(Q_comp[0, 1], e_formula), f"Composition formula mismatch for ({e1}, {e2})"
    assert np.isclose(Q_comp[1, 0], e_formula), f"Composition formula mismatch for ({e1}, {e2})"
print(f"PASS: BSC composition formula verified for {len(test_pairs)} pairs")

# 4. Composed BSC is itself a BSC (symmetric)
e1, e2 = 0.1, 0.2
Q_comp = compose_channels(make_bsc(e1), make_bsc(e2))
assert np.isclose(Q_comp[0, 0], Q_comp[1, 1]), "Composed BSC not symmetric"
assert np.isclose(Q_comp[0, 1], Q_comp[1, 0]), "Composed BSC not symmetric"
print("PASS: Composed BSC is itself symmetric (a BSC)")

# 5. BSC capacity at boundary values
assert np.isclose(bsc_capacity(np.array([0.5]))[0], 0.0), "Capacity at eps=0.5 should be 0"
assert bsc_capacity(np.array([0.001]))[0] > 0.98, "Capacity near eps=0 should be near 1"
print("PASS: BSC capacity boundary values correct")

# 6. Z-channel has omega = 0
Q_z = make_asymmetric(epsilon=0.05, omega=0.0)
assert np.isclose(Q_z[0, 1], 0.0), "Z-channel should have zero FNR (omega=0)"
print("PASS: Z-channel has omega = 0 (no false negatives)")

print()
print("All Chapter 2 verification checks passed.")

---

## Next Steps

The companion notebook for **Chapter 3** (Boolean Algebra Under Noise) will build
on the channel framework introduced here, developing the composition theorem for
chained identities-with-error and connecting it to Boolean algebra over noisy channels.

When it exists, it will be at: `python/notebooks/ch03-boolean-algebra-under-noise.ipynb`.

**Chapter 4 (Kronecker Product and Independent Queries)** establishes the
*parallel* analog of the serial channel composition in Section 2.3:
the Kronecker product $Q_n = Q^{\otimes n}$ for $n$ independent queries.
Its notebook: `python/notebooks/ch04-kronecker.ipynb`.